## はじめに

このノートブックでは、GlacierStyle ECサイトの各種データを加工・変換し、分析可能な形式に整形します。

**主な処理内容:**
- SNSログの感情分析・分類
- 音声ログの要約・マスキング
- 広告クリエイティブの分析
- FAQドキュメントのチャンク化
- 運用マニュアルのチャンク化
- テーブルメタデータの自動生成

In [104]:
%%sql -r result_env_setup
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/part2/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(15, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
display_image('architecture.png')

## 2. データの加工・変換

このセクションでは、Raw層のデータをAI機能を活用して加工・変換し、Silver/Gold層のテーブルを作成します。

### 2-1. SNSログの分類・感情分析

SNSメンション（Twitter/Instagram等）の生データに対して、以下のAI処理を実施:
- **AI_EXTRACT**: 商品名・カテゴリ・問い合わせタイプの抽出
- **AI_SENTIMENT**: 投稿の感情分析（ポジティブ/ネガティブ/ニュートラル）
- **AI_CLASSIFY**: 投稿カテゴリの分類（称賛/クレーム/質問/提案）

In [ ]:
display_image('ai_extract.png')

In [ ]:
display_image('ai_sentiment.png')

In [ ]:
display_image('ai_classify.png')

In [106]:
%%sql -r result_silver_sns_analyzed
-- ============================================================================
-- SNS生ログのAI分析とGold層への保存
-- ============================================================================
-- Gold層テーブルの作成
CREATE OR REPLACE TABLE gold_sns_mentions_analyzed AS
WITH extracted_data AS (
    SELECT 
        post_id,
        platform,
        post_type,
        username,
        display_name,
        content,
        posted_at,
        likes,
        retweets,
        replies,
        hashtags,
        mentioned_products,
        media_urls,
        
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            content,
            OBJECT_CONSTRUCT(
                'product_name', 'mentioned product name',
                'category', 'product category (e.g., ファッション, インテリア, テック)',
                'inquiry_type', 'inquiry type (e.g., 質問, レビュー, クレーム, 称賛, 提案)'
            )
        ) AS extracted_info
        
    FROM raw_sns_mentions
),
sentiment_analysis AS (
    SELECT 
        *,
        SNOWFLAKE.CORTEX.AI_SENTIMENT(content) AS sentiment_result
    FROM extracted_data
),
classified_data AS (
    SELECT 
        *,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            content,
            ['称賛', 'クレーム', '質問', '提案']
        ) AS classification_result
    FROM sentiment_analysis
)
SELECT 
    post_id,
    platform,
    post_type,
    username,
    display_name,
    content,
    posted_at,
    likes,
    retweets,
    replies,
    hashtags,
    mentioned_products,
    media_urls,
    
    -- AI抽出情報
    extracted_info:response.product_name::VARCHAR AS extracted_product_name,
    extracted_info:response.category::VARCHAR AS extracted_category,
    extracted_info:response.inquiry_type::VARCHAR AS inquiry_type,
    
    -- 感情分析結果
    sentiment_result:categories[0].name::VARCHAR AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR AS sentiment,
    
    -- カテゴリ分類結果
    classification_result:labels[0]::VARCHAR AS post_category,

    -- メタデータ
    CURRENT_TIMESTAMP() AS processed_at
    
FROM classified_data;

In [107]:
%%sql -r preview_silver_sns
-- ============================================================================
-- 分析結果の確認
-- ============================================================================
-- Gold層テーブルの内容をプレビュー
SELECT * FROM gold_sns_mentions_analyzed LIMIT 10;

### 2-2. 音声ログの要約・分類・マスキング

コールセンター音声ログの文字起こしデータに対して、以下のAI処理を実施:
- **AI_REDACT**: 個人情報（氏名・電話番号・住所・クレカ番号）の自動マスキング
- **AI_SENTIMENT**: 顧客感情の分析
- **AI_CLASSIFY**: 問い合わせカテゴリ分類
- **AI_AGG**: 通話内容の要約生成

In [ ]:
display_image('ai_redact.png')

In [ ]:
display_image('ai_agg.png')

In [ ]:
display_image('ai_agg_2.png')

In [108]:
%%sql -r result_agg_voc_summary
-- ============================================================================
-- 音声ログのAI分析とGold層への保存
-- ============================================================================
CREATE OR REPLACE TABLE gold_voice_logs AS
SELECT
    -- 元のカラム（transcribed_text以外）
    * EXCLUDE transcribed_text,
    
    -- 1. AI_REDACTによる個人情報のマスキング
    -- 氏名、電話番号、住所、クレジットカード番号を自動検出・マスキング
    SNOWFLAKE.CORTEX.AI_REDACT(
        transcribed_text
    )::VARCHAR AS transcribed_text_masked,
    
    -- 2. AI_SENTIMENTによる顧客感情の分析
    -- マスキング済みテキストに対して感情分析を実施
    SNOWFLAKE.CORTEX.AI_SENTIMENT(
        transcribed_text_masked
    ) AS sentiment_result,
    
    -- 感情の詳細を抽出
    sentiment_result:categories[0].name::VARCHAR AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR AS sentiment,
    
    -- 3. AI_CLASSIFYによる問い合わせカテゴリ分類
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        transcribed_text_masked,
        ['商品に関する問い合わせ', '配送に関する問い合わせ', '返品・交換', '決済・支払い', 'アカウント・会員登録', 'クレーム', 'その他']
    ) AS classification_result,
    
    -- 分類の詳細を抽出
    classification_result:labels[0]::VARCHAR AS inquiry_category,
    
    -- 4. AI_AGGによる通話内容の要約生成（GROUP BY対象）
    AI_AGG(
        transcribed_text_masked, 
        '音声ログを400文字以内で要約してください。顧客の主な問い合わせ内容、要望、および解決状況を含めてください。'
    ) AS transcribed_text_summary,
    
    -- メタデータ
    CURRENT_TIMESTAMP() AS processed_at

FROM raw_voice_logs
GROUP BY ALL;

In [ ]:
%%sql -r dataframe_3
-- ============================================================================
-- 音声ログGold層の確認
-- ============================================================================
SELECT * FROM gold_voice_logs LIMIT 10;

### 2-3. 広告メタデータの抽出

広告クリエイティブデータに対して、以下のAI処理を実施:
- **AI_COMPLETE**: 画像からビジュアル要素を抽出（色・構図・人物・印象等）
- **AI_CLASSIFY**: 画像スタイルの分類
- **AI_EXTRACT**: コピーテキストから訴求ポイント・CTA・キーワード等を抽出
- **AI_SENTIMENT**: コピーの感情分析

In [ ]:
display_image('ai_complete.png')

In [83]:
%%sql -r result_silver_ad_visual_temp
CREATE OR REPLACE TEMP TABLE silver_ad_visual_analysis AS
WITH image_files AS (
    SELECT 
        relative_path,
        SPLIT_PART(relative_path, '/', -1) AS file_name,
        file_url,
        size,
        last_modified
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path ILIKE 'ad_images/%.png'
       OR relative_path ILIKE 'ad_images/%.jpg'
       OR relative_path ILIKE 'ad_images/%.jpeg'
)
SELECT 
    file_name,
    relative_path,
    file_url,
    size,
    last_modified,
    
    -- AI_COMPLETEで画像からビジュアル要素を抽出
    AI_COMPLETE(
        'openai-gpt-4.1',
        '以下の広告画像を分析し、JSON形式で以下の要素を抽出してください：
        {
            "メインカラー": "画像の主要な色（最大3色）",
            "構図タイプ": "中央配置/三分割/対角線/その他",
            "人物有無": "あり/なし",
            "人物の特徴": "性別、年齢層、表情など（人物がいる場合）",
            "商品配置": "商品の位置と見せ方",
            "背景スタイル": "単色/グラデーション/写真/イラストなど",
            "全体的な印象": "高級感/カジュアル/モダン/ナチュラルなど",
            "ターゲット層推定": "想定されるターゲット層"
        }
        JSON形式のみで回答してください。',
        TO_FILE('@DATA_STAGE', relative_path)
        
    ) AS visual_analysis_raw,
    TRY_PARSE_JSON(visual_analysis_raw) visual_analysis_raw_json,
    
    -- 画像分類（スタイル別）
    AI_CLASSIFY(
        TO_FILE('@DATA_STAGE', relative_path),
        ['商品フォーカス', 'ライフスタイル', 'テキスト中心', 'キャンペーン訴求', 'ブランドイメージ']
    ) AS image_style_classification,
    
    -- 分類結果を抽出
    image_style_classification:labels[0]::VARCHAR AS image_style,
    
    CURRENT_TIMESTAMP() AS analyzed_at

FROM image_files;

In [84]:
%%sql -r result_silver_ad_copy
CREATE OR REPLACE TEMP TABLE silver_ad_copy_analysis AS
SELECT 
    creative_id,
    creative_name,
    copy_text,
    headline,
    cta_text,
    
    -- AI_EXTRACTでコピー要素を抽出
    SNOWFLAKE.CORTEX.AI_EXTRACT(
        copy_text || ' ' || headline || ' ' || cta_text,
        OBJECT_CONSTRUCT(
            'appeal_type', 'main appeal point (e.g., 品質訴求, 価格訴求, 限定訴求, 感情訴求)',
            'cta_type', 'call-to-action type (e.g., 購入促進, 情報収集, 会員登録)',
            'keywords', 'main keywords (comma separated)',
            'target_emotion', 'target emotion (e.g., 安心感, ワクワク感, 緊急感)',
            'usp', 'unique selling proposition'
        )
    ) AS copy_extraction,
    
    -- 抽出結果を個別カラムに展開
    copy_extraction:response.appeal_type::VARCHAR AS appeal_type,
    copy_extraction:response.cta_type::VARCHAR AS cta_type,
    copy_extraction:response.keywords::VARCHAR AS keywords,
    copy_extraction:response.target_emotion::VARCHAR AS target_emotion,
    copy_extraction:response.usp::VARCHAR AS usp,
    
    -- コピーの感情分析
    SNOWFLAKE.CORTEX.AI_SENTIMENT(copy_text) AS copy_sentiment,
    copy_sentiment:categories[0].name::VARCHAR AS sentiment,
    copy_sentiment:categories[0].sentiment::VARCHAR AS sentiment_score,
    
    -- コピーのスタイル分類
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        copy_text,
        ['直接的訴求', '感情的訴求', '論理的訴求', '限定・緊急訴求', 'ストーリー訴求']
    ) AS copy_style_result,
    copy_style_result:labels::VARCHAR AS copy_style,
    
    CURRENT_TIMESTAMP() AS analyzed_at

FROM raw_ad_creatives;


In [89]:
%%sql -r result_gold_ad_creative
CREATE OR REPLACE TABLE gold_ad_creative_analysis AS
SELECT 
    -- 広告クリエイティブ基本情報
    c.creative_id,
    c.creative_name,
    c.creative_type,
    c.campaign_id,
    c.platform,
    c.target_segment,
    
    -- テキスト関連
    c.copy_text,
    c.headline,
    c.cta_text,
    
    -- コピー分析結果
    t.appeal_type,
    t.cta_type,
    t.keywords,
    t.target_emotion,
    t.usp,
    t.copy_style,
    t.sentiment,
    
    -- パフォーマンスデータ
    c.impressions,
    c.clicks,
    c.conversions,
    c.spend,
    CASE WHEN c.impressions > 0 THEN (c.clicks::FLOAT / c.impressions) * 100 ELSE 0 END AS ctr,
    CASE WHEN c.clicks > 0 THEN (c.conversions::FLOAT / c.clicks) * 100 ELSE 0 END AS cvr,
    CASE WHEN c.conversions > 0 THEN c.spend / c.conversions ELSE 0 END AS cpa,

    -- 画像データ
    v.visual_analysis_raw_json,
    v.image_style,
    
    -- メタデータ
    CURRENT_TIMESTAMP() AS processed_at

FROM raw_ad_creatives c
LEFT JOIN silver_ad_copy_analysis t ON c.creative_id = t.creative_id
LEFT JOIN silver_ad_visual_analysis v ON SPLIT_PART(c.image_file_path, '/', -1) = SPLIT_PART(v.relative_path, '/', -1) 

In [ ]:
%%sql -r dataframe_2
-- ============================================================================
-- 広告データ Gold層の確認
-- ============================================================================
SELECT * FROM gold_ad_creative_analysis LIMIT 10;

### 2-4. FAQドキュメントのチャンク化

FAQドキュメントをSPLIT_TEXT_MARKDOWN_HEADERでチャンク化後、カテゴリ別に集約し、AI_AGGで要約を生成:
- SPLIT_TEXT_MARKDOWN_HEADERでチャンク化
- カテゴリ別の質問項目サマリーをJSON形式で出力
- RAG検索用のコンテキスト情報として活用

In [93]:
%%sql -r preview_raw_faq
-- ============================================================================
-- FAQドキュメントデータの確認
-- ============================================================================
SELECT * FROM raw_faq_documents LIMIT 100;

In [99]:
%%sql -r result_gold_faq

-- ============================================================================
-- FAQドキュメントのGold層テーブル作成
-- ============================================================================
-- カテゴリ別にAI_AGGで質問項目を要約し、元データと結合
CREATE OR REPLACE TABLE gold_faq_documents AS
WITH cte AS(
    SELECT 
        t.relative_path,
        t.file_url,
        t.size,
        t.last_modified,
        t2.value AS raw_value,
        t2.value:headers:header_1::VARCHAR AS chapter,
        t2.value:headers:header_2::VARCHAR AS section,
        t2.value:headers:header_3::VARCHAR AS paragraph,
        t2.value:headers:header_4::VARCHAR AS subcategory,
        t2.value:chunk::TEXT AS content_chunk
    FROM raw_faq_documents t,
    LATERAL FLATTEN(INPUT => 
        SNOWFLAKE.CORTEX.SPLIT_TEXT_MARKDOWN_HEADER(
            t.contents:content, 
            OBJECT_CONSTRUCT('#', 'header_1', '##', 'header_2', '###', 'header_3', '####', 'header_4'),
            10000
        )
    ) t2
),

category_summary AS (
    SELECT
        chapter,
        section,
        AI_AGG(
            content_chunk,
            'どういった質問項目があるのかを中心に要約してください。出力形式はJSON形式にしてください'
        ) AS summary_category
    FROM cte
    GROUP BY ALL
)

SELECT
    t1.* exclude (chapter, section, paragraph, subcategory),
    t2.summary_category
FROM cte AS t1
LEFT OUTER JOIN category_summary AS t2
    ON t1.chapter = t2.chapter AND
       t1.section = t2.section

In [ ]:
%%sql -r dataframe_5
-- ============================================================================
-- FAQマニュアルGold層の確認
-- ============================================================================
SELECT * FROM gold_faq_documents LIMIT 100;

### 2-5. 運用マニュアルのチャンク化

運用マニュアルをチャンク化し、画像を抽出:
- **SPLIT_TEXT_MARKDOWN_HEADER**: マークダウン形式でチャンク化
- **AI_PARSE_DOCUMENT**: PDFから画像をBase64形式で抽出
- **SAVE_EXTRACTED_IMAGES**: 抽出した画像をステージに保存するプロシージャ

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE PROCEDURE SAVE_EXTRACTED_IMAGES(config OBJECT)
RETURNS TABLE(status STRING, message STRING)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
AS
$$
import base64
import io
import os
from snowflake.snowpark.types import StructType, StructField, StringType

def process_parse_document_result(data: dict):
    """AI_PARSE_DOCUMENTの結果から画像情報を抽出"""
    images = data.get("images", [])
    if images:  # page_split: false の場合
        for image in images:
            id = image["id"]
            data_field, image_base64 = image["image_base64"].split(";", 1)
            extension = data_field.split("/")[1]
            base64_data = image_base64.split(",")[1]
            yield id, extension, base64_data
    else:  # page_split: true の場合
        pages = data.get("pages", [])
        for page in pages:
            page_images = page.get("images", [])
            for image in page_images:
                id = image["id"]
                data_field, image_base64 = image["image_base64"].split(";", 1)
                extension = data_field.split("/")[1]
                base64_data = image_base64.split(",")[1]
                yield id, extension, base64_data

def run(session, config):
    destination_path = config["DESTINATION_PATH"]
    parse_result = config["PARSE_DOCUMENT_RESULT"]
    
    saved_count = 0
    for image_id, extension, base64_data in process_parse_document_result(parse_result):
        image_bytes = base64.b64decode(base64_data)
        name_without_ext = os.path.splitext(image_id)[0]
        file_path = f"{destination_path}/{name_without_ext}.{extension}"
        session.file.put_stream(
            input_stream=io.BytesIO(image_bytes),
            stage_location=file_path,
            auto_compress=False,
            overwrite=True
        )
        saved_count += 1
    
    schema = StructType([
        StructField("STATUS", StringType()),
        StructField("MESSAGE", StringType())
    ])
    return session.create_dataframe([["SUCCESS", f"Saved {saved_count} images"]], schema=schema)
$$;

In [ ]:
-- ============================================================================
-- 抽出した画像をステージに保存
-- ============================================================================
-- SAVE_EXTRACTED_IMAGESプロシージャを呼び出して、
-- AI_PARSE_DOCUMENTで抽出した画像をステージに保存
CALL SAVE_EXTRACTED_IMAGES(
    OBJECT_CONSTRUCT(
        'DESTINATION_PATH', '@EXTRACTED_IMAGES_STAGE',
        'PARSE_DOCUMENT_RESULT', (SELECT contents FROM raw_operation_manuals)
    )
);

LIST @EXTRACTED_IMAGES_STAGE;

In [102]:
-- ============================================================================
-- 運用マニュアルのGold層テーブル作成
-- ============================================================================
-- 部門別にAI_AGGで章・節を要約し、元データと結合
CREATE OR REPLACE TABLE gold_operation_manuals AS
SELECT 
    t.relative_path,
    t.file_url,
    t.size,
    t.last_modified,
    -- t2.value AS raw_value,
    -- t2.value:headers:header_1::VARCHAR AS department,
    -- t2.value:headers:header_2::VARCHAR AS chapter,
    -- t2.value:headers:header_3::VARCHAR AS section,
    -- t2.value:headers:header_4::VARCHAR AS subsection,
    t2.value:chunk::TEXT AS content_chunk
FROM raw_operation_manuals t,
LATERAL FLATTEN(INPUT => 
    SNOWFLAKE.CORTEX.SPLIT_TEXT_MARKDOWN_HEADER(
        t.contents:content, 
        OBJECT_CONSTRUCT('#', 'header_1', '##', 'header_2', '###', 'header_3', '####', 'header_4'),
        10000
    )
) t2;

In [103]:
%%sql -r preview_gold_operation_manuals
-- ============================================================================
-- 運用マニュアルGold層の確認
-- ============================================================================
SELECT * FROM gold_operation_manuals LIMIT 100;

### 2-6. 商品マスタとの突合（名寄せ）

SNSメンションから抽出した商品名と商品マスタをAI_SIMILARITYで突合:
- カテゴリでフィルタ後、商品名の類似度を計算
- 類似度が高い順にソート

In [ ]:
display_image('ai_similarity.png')

In [ ]:
display_image('embedding.png')

In [122]:
%%sql -r result_product_matching
-- ============================================================================
-- SNSメンションと商品マスタの突合（名寄せ）
-- ============================================================================
-- AI_SIMILARITYを使用して商品名の類似度を計算
CREATE OR REPLACE TABLE gold_sns_mentions_with_product_master AS
WITH product_match AS (
    SELECT 
        *
    FROM gold_sns_mentions_analyzed t1
    INNER JOIN dim_products t2
        ON t2.category_l1 = t1.extracted_category
)
SELECT
    *,
    AI_SIMILARITY(extracted_product_name, product_name) AS similarity
FROM product_match
ORDER BY post_id, similarity DESC;

In [124]:
%%sql -r dataframe_1
-- ============================================================================
-- 商品マスタ突合結果の確認
-- ============================================================================
SELECT * FROM gold_sns_mentions_with_product_master ORDER BY post_id LIMIT 1000;

## まとめ

このノートブックでは、GlacierStyle ECサイトのRaw層データをSnowflake Cortex AI機能を活用して加工・変換し、分析可能なSilver/Gold層テーブルを作成しました。

### 作成したテーブル一覧

**Silver層（一時テーブル）**
- `silver_ad_visual_analysis`: 広告画像のビジュアル分析結果
- `silver_ad_copy_analysis`: 広告コピーのテキスト分析結果

**Gold層**
- `gold_sns_mentions_analyzed`: SNSメンションのAI分析結果（感情・分類・抽出情報）
- `gold_voice_logs`: 音声ログの要約・マスキング・分類結果
- `gold_ad_creative_analysis`: 広告クリエイティブの統合分析結果
- `gold_faq_documents`: FAQドキュメントのカテゴリ別要約
- `gold_operation_manuals`: 運用マニュアルの部門別要約
- `gold_sns_mentions_with_product_master`: SNSメンションと商品マスタの突合結果

### 使用したCortex AI関数

- `AI_EXTRACT`: テキストから構造化情報を抽出
- `AI_SENTIMENT`: 感情分析（ポジティブ/ネガティブ/ニュートラル）
- `AI_CLASSIFY`: テキスト・画像のカテゴリ分類
- `AI_REDACT`: 個人情報の自動マスキング
- `AI_AGG`: テキストの要約生成
- `AI_COMPLETE`: 画像分析・自由形式のAI処理
- `AI_SIMILARITY`: テキスト間の類似度計算
- `AI_GENERATE_TABLE_DESC`: テーブル・カラム説明の自動生成
- `TRANSLATE`: 多言語翻訳

### 次のステップ

- **Part 3**: 各種テーブルへのメタデータ自動付与